In [59]:
from dotenv import load_dotenv
load_dotenv()

True

In [60]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate

Data Load

In [61]:
loader = PyPDFLoader("../data/medical_report.pdf")

pdf = loader.load()

len(pdf)


9

Data Chunks

In [62]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 20)
splitted_data = splitter.split_documents(pdf)
print(len(splitted_data))

25


Embeddings

In [48]:
embed = OpenAIEmbeddings(model="text-embedding-3-large")
store_vector = Chroma.from_documents(
    documents=splitted_data,
    embedding=embed
)


Similarity Search

In [63]:
query = "ML and DS content"
result = store_vector.similarity_search(query)
print(result)

[Document(id='a097a77c-a6a6-4b1b-8773-88b802f6fca0', metadata={'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/137.0.0.0 Safari/537.36', 'page': 4, 'source': '../data/data_science_syllabus.pdf', 'page_label': '5', 'title': '📘 1-Year Roadmap: Data Analytics, Data Science & GenAI', 'moddate': '2025-12-13T08:07:26+00:00', 'creationdate': '2025-12-13T08:07:26+00:00', 'total_pages': 10, 'producer': 'Skia/PDF m137'}, page_content='🎤  Mock Interview: Stats Scenarios \ue09d Probabilities \ue09d Tests\n✅  Module 6: Machine Learning – I (Supervised \nLearning) (4 weeks)\nDuration: Month 6\nTopics:\nML pipeline\nRegression: Linear, Logistic\nDecision Tree, Random Forest, KNN\nTrain-test split, model evaluation\nTools:\nScikit-learn, Google Colab, ChatGPT, PyCaret (optional)\nMini Project:\nLoan Approval or House Price Prediction\nPredict Diabetes from health dataset\n🎤  Mock Interview: Supervised Learning Models \ue09d Metrics\n✅  Module 7: Machin

In [50]:
context = ""
for doc in result:
    context += doc.page_content + "\n"

print(context)

🎤  Mock Interview: Stats Scenarios  Probabilities  Tests
✅  Module 6: Machine Learning – I (Supervised 
Learning) (4 weeks)
Duration: Month 6
Topics:
ML pipeline
Regression: Linear, Logistic
Decision Tree, Random Forest, KNN
Train-test split, model evaluation
Tools:
Scikit-learn, Google Colab, ChatGPT, PyCaret (optional)
Mini Project:
Loan Approval or House Price Prediction
Predict Diabetes from health dataset
🎤  Mock Interview: Supervised Learning Models  Metrics
✅  Module 7: Machine Learning – II (Unsupervised & 
Feature Engineering) (3 weeks)
Duration: Month 7
Topics:
KMeans Clustering
Dimensionality Reduction: PCA
Feature selection, encoding, scaling
Model tuning GridSearchCV
Tools:
Scikit-learn, Seaborn, Colab
Mini Project:
📘  1Year Roadmap: Data Analytics, Data Science & GenAI
5
Customer Segmentation
E-commerce Product Clustering
🎤  Mock Interview: Unsupervised Learning  Features
✅  Module 8: Time Series + Intro to Deep Learning (3 
weeks)
Duration: Month 8
Topics:
Time Se

Now create LLM

In [16]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model = 'gpt-4o')

res = llm.invoke(f"""Can you provide me answers based on my context {context} and question {query}""")

In [17]:
print(res.content)

Given your context and curriculum, let me provide some insights and answer potential questions you might encounter during your studies or interviews related to Machine Learning (ML) and Data Science (DS).

### Module 6: Supervised Learning

1. **ML Pipeline:**
   - **Question:** What are the main steps in a typical machine learning pipeline?
   - **Answer:** A typical ML pipeline includes data collection, data cleaning, exploratory data analysis, feature engineering, model selection, training, evaluation, and deployment.

2. **Regression Models:**
   - **Question:** How do linear regression and logistic regression differ?
   - **Answer:** Linear regression is used for predicting a continuous target variable, while logistic regression is used for binary classification problems and predicts the probability that a given input belongs to a category.

3. **Model Evaluation:**
   - **Question:** What metrics would you use to evaluate a regression model?
   - **Answer:** Common metrics includ

##### Proper Project / Company-Standard

In [18]:
## for proper standards , we need chain because everything should be in a pipeline to automate process.

In [ ]:
## overall context-generate -> Prompt -> LLM -> strparser

In [37]:
def get_context(query:str):
    data = store_vector.similarity_search(query=query)
    context = ""
    for doc in data:
        context += doc.page_content + "\n"

    return {"context":context, "query":query}




In [38]:
prompts = PromptTemplate.from_template("""
you are an AI agent.
Context: {context}
query: {query}
""")


In [39]:
rag_chains = get_context | prompts | llm 



In [ ]:
res = rag_chains.invoke("who referred these blood test?")

print(res.content)

The blood tests for Ms. Nikita Choudhary were referred by Dr. Nitin Nahar, as indicated in the report.


In [58]:
res2 = rag_chains.invoke("what is the level of RBC count and is it under range?")
print(res2.content)

The RBC (Red Blood Cell) Count is reported as 4.47 million/mm³. The biological reference interval provided for RBC Count is 3.80 - 4.80 million/mm³. Thus, the RBC Count of 4.47 million/mm³ is within the normal range.
